In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import statsmodels.api as sm
import pickle as pkl
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn import metrics
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

In [8]:
num_group = 1

df_1 = pkl.load(open('..\data_project\df_task_1_group_'+str(num_group)+'.pkl', 'rb'))
df_1.head()

<>:3: SyntaxWarning: invalid escape sequence '\d'
<>:3: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Eshel\AppData\Local\Temp\ipykernel_29564\1190121623.py:3: SyntaxWarning: invalid escape sequence '\d'
  df_1 = pkl.load(open('..\data_project\df_task_1_group_'+str(num_group)+'.pkl', 'rb'))


,age,expirience,hour,worker_rank,days_c,toys_c,factories_rand_c,months_c,safety_event
0,47,3,11,1,Friday,toy_cars,Huainan,January,0
1,44,9,2,4,Friday,puzzels,Hefei,March,0
2,44,2,8,1,Friday,toy_cars,Huangshan,March,0
3,46,1,1,0,Friday,puzzels,Huangshan,December,1
4,47,5,4,2,Thursday,puzzels,Huangshan,November,1


<div dir="rtl">

### ברוכים הבאים לפרויקט גמר
    
### בחלק הראשון נעסוק בתאונות של עובדים במפעלי צעצועים בסין. 
    
### במפעלים אלו הפועלים עובדים שעות ארוכות עם מכונות מסוכנות והרבה מהם נפצעים במהלך עבודתם. 
    
### בחלק הראשון של הפרויקט נרצה למצוא את ההסתברות לפציעה של עובד.
    
### יש לנו מספר משתנים מסבירים: 

In [3]:
df_1.columns

Index(['age', 'expirience', 'hour', 'worker_rank', 'days_c', 'toys_c',
       'factories_rand_c', 'months_c', 'safety_event'],
      dtype='object')

* age: the worker age (numeric)
* expirience: the number of years the worker is in the factory (numeric)
* hour: the hour of the shift: 1-12 (numeric)
* worker_rank: workers are being ranked by their employer, the higher the rank the most senior the worker is. Ranks are between 0 to 5 (numeric)
* days: day of the week (categorical)
* toys: the type of the toy (categorical)
* factory: the factory location refers to a city in china (categorical)
* month: the month in the year (categorical)
* safety_event: 1 if an accidents accured, 0 if not.

<div dir="rtl">

## לפניך מספר מטלות: 
    
### 1. להציג scatter plot עבור כל זוג משתנים מסבירים רציפים (לא קטגוריאלים) ולחשב קורלציה (r) בין כל זוג. האם יש זוג משתנים שיש בינהם קשר? אם כן, אילו? 

### 2. בחרו שני משתנים קטגוריאליים. 
### הציגו בשני גרפים שונים את מספר העובדים שעברו תאונה לעומת אלו שלא עברו תאונה (בצבעים שונים תאונה לעומת לא תאונה), עבור כל אחת מהקטגוריות בנפרד. 
### עבור כל אחד מהמשתנים בדקו האם אתם מזהים קטגוריה מסוימת שעבורה יש יותר סיכוי לתאונה בהשוואה לקטגוריות האחרות שבאותו משתנה? 
### אם כן, מהי אותה קטגוריה בכל אחד מהמשתנים שבחרתם? 
### בהתחשב בתשובתכם, מה לדעתכם הסבירות שהמקדמים של המשתנים שבחרתם יהיו מובהקים כחלק מרגרסיה לוגיסטית?
    
### 3. הציגו בגרף של שני עמודות את מספר התאונות לעומת לא-תאונות עבור כלל העובדים. חשבו את השכיחות היחסית לתאונה וללא תאונה.
    
### 4. נא להוציא הסטוגרמה של אחד המשתנים הרציפים. יש להסביר במשפט מה הנתונים מה מציגים (טווח, היכן יש ריכוז גבוה וכו')
    
### 5. לפני שמתחילים את הניתוח הלוגיסטי - נא לפצל את כל הנתונים לtrain -ו test לפי יחס של 30-70
    
### 6. יש למצוא את המודל של הרגרסיה הלוגיסטית עבור המודל המתקבל מstepwise selection. נא להשוות ערכים של AIC וBIC נגד המודל בו כל המשתנים נכנסים למודל.     

## עבור סעיפים 7-11 נא להשתמש במודל שהתקבל על ידי הstepwise selection.
    
### 7. מה הסיכוי לתאונה עבור מצב שכל משתני הדמה במצב בסיס שלהם וכל המשתנים הרציפים עם הערך 2?
    
### 8. נרצה כעת להשוות בין 2 קבוצות שונות של משתנה קטוגיריאלי כפונקציה של כל הערכים האפשריים של משתנה רציף כלשהו. יש לחשב את ההסתברות לתאונה עבור כל אחת מהקבוצות כאשר הערכים של שאר המשתנים קבועים. עבור שאר המשתנים הקטוגוריאלים תזינו למודל את הערך 0  ואילו עבור משתנים רציפים אחרים תזינו את ההממוצע מהtrain set. 
    
### 9. עם אותם נתונים של סעיף 8, יש לחשב את הlog odds ratio בין 2 הקבוצות של המשתנה הקטגוריאלי שבחרתם עבור הרבה ערכים של המשתנה הרציף. כיצד הגרף נראה? נא להסביר את התוצאה? מה משמעות הערך שקיבלתם בגרף?
    
### סעיפים 10-12 מתייחסים לtest set.    

### 10. מהו הערך סף המינימלי כך שה-sensitivity יהיה לפחות 0.8? יש לעשות זאת עבור הtest set?
    
### 11. יש להוציא ROC curve עבור המודל הסופי ולחשב את הAUC. 

### 12. יש להוציא עוד שני ROC curves. פעם אחת כאשר אתם מוציאים משתנה רציף אחד ופעם שנייה כאשר אתם מוציאים משתנה קטגוריאלי. יש להכניס את כל הROC curves כולל את העקומה מסעיף 11 ולשים בגרף אחד. יש להשוות בין שלוש ה-AUC? מה קרה לAUC אחרי שהוצאתם את המשתנה הרציף ואחרי שהוצאתם את המשתנה הקטוגוריאלי? מי הושפע יותר ? מה זה אומר לדעתכם?
    
    
### בהצלחה
    
    